# Notebook 4 – Date & Time Feature Engineering

## Learning objectives
Extract calendar features, create elapsed-time and customer-lifecycle features, and understand recency, frequency, and seasonality. Always define the reference date and ensure the feature uses only information available at prediction time.

## 1. Sample Dataset

### Example
An e-commerce dataset contains customer birth dates, signup dates, order dates, and last-purchase dates. These dates can be converted into useful numeric and categorical features.

In [1]:
import pandas as pd
import numpy as np

reference_date = pd.Timestamp("2026-11-05")

orders = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105],
    "date_of_birth": ["1998-04-12", "1985-09-30", "2001-01-20", "1992-06-15", "1978-12-02"],
    "signup_date": ["2024-01-10", "2023-06-20", "2025-03-01", "2022-11-15", "2026-01-05"],
    "order_date": ["2026-11-01", "2026-10-28", "2026-10-15", "2026-11-05", "2026-09-20"],
    "last_purchase_date": ["2026-11-01", "2026-10-28", "2026-10-15", "2026-11-05", "2026-09-20"],
    "order_amount": [1200, 850, 450, 2300, 700]
})

for column in ["date_of_birth", "signup_date", "order_date", "last_purchase_date"]:
    orders[column] = pd.to_datetime(orders[column])

orders

,customer_id,date_of_birth,signup_date,order_date,last_purchase_date,order_amount
0,101,1998-04-12,2024-01-10,2026-11-01,2026-11-01,1200
1,102,1985-09-30,2023-06-20,2026-10-28,2026-10-28,850
2,103,2001-01-20,2025-03-01,2026-10-15,2026-10-15,450
3,104,1992-06-15,2022-11-15,2026-11-05,2026-11-05,2300
4,105,1978-12-02,2026-01-05,2026-09-20,2026-09-20,700


### Output explanation
The date columns are converted to Pandas datetime values. Datetime conversion is necessary before extracting calendar components or calculating time differences.

## 2. Year, Month, and Day

These components can reveal yearly trends, monthly demand, or within-month patterns.

### Example
Extract year, month, and day from each order date.

In [2]:
orders["order_year"] = orders["order_date"].dt.year
orders["order_month"] = orders["order_date"].dt.month
orders["order_day"] = orders["order_date"].dt.day
orders[["order_date", "order_year", "order_month", "order_day"]]

,order_date,order_year,order_month,order_day
0,2026-11-01,2026,11,1
1,2026-10-28,2026,10,28
2,2026-10-15,2026,10,15
3,2026-11-05,2026,11,5
4,2026-09-20,2026,9,20


### Output explanation
The new columns separate the calendar date into numeric components. Use them when the calendar pattern has business meaning.

## 3. Day of Week and Week Number

Day of week can capture weekday/weekend purchasing behaviour. Week number can help identify weekly patterns.

### Example
Extract weekday name and ISO week number.

In [3]:
orders["day_of_week"] = orders["order_date"].dt.day_name()
orders["week_number"] = orders["order_date"].dt.isocalendar().week.astype(int)
orders[["order_date", "day_of_week", "week_number"]]

,order_date,day_of_week,week_number
0,2026-11-01,Sunday,44
1,2026-10-28,Wednesday,44
2,2026-10-15,Thursday,42
3,2026-11-05,Thursday,45
4,2026-09-20,Sunday,38


### Output explanation
The weekday is a label and the week number is an integer. For cyclic patterns, consider cyclical encoding so Sunday and Monday are treated as neighbouring days.

## 4. Quarter

A quarter groups months into four three-month periods.

### Example
Extract the calendar quarter.

In [4]:
orders["quarter"] = orders["order_date"].dt.quarter
orders[["order_date", "quarter"]]

,order_date,quarter
0,2026-11-01,4
1,2026-10-28,4
2,2026-10-15,4
3,2026-11-05,4
4,2026-09-20,3


### Output explanation
Quarter values range from 1 to 4. Quarter can help a model represent broad seasonal or reporting-period patterns.

## 5. Weekend Indicator

A weekend indicator marks whether an event occurred on Saturday or Sunday.

### Example
Create a Boolean weekend feature.

In [5]:
orders["is_weekend"] = orders["order_date"].dt.dayofweek >= 5
orders[["order_date", "day_of_week", "is_weekend"]]

,order_date,day_of_week,is_weekend
0,2026-11-01,Sunday,True
1,2026-10-28,Wednesday,False
2,2026-10-15,Thursday,False
3,2026-11-05,Thursday,False
4,2026-09-20,Sunday,True


### Output explanation
`True` means Saturday or Sunday. This can be useful when customer behaviour differs between weekdays and weekends.

## 6. Month Start and Month End

These indicators identify whether a date falls on the first or last calendar day of a month.

### Example
Create month-start and month-end flags.

In [6]:
orders["is_month_start"] = orders["order_date"].dt.is_month_start
orders["is_month_end"] = orders["order_date"].dt.is_month_end
orders[["order_date", "is_month_start", "is_month_end"]]

,order_date,is_month_start,is_month_end
0,2026-11-01,True,False
1,2026-10-28,False,False
2,2026-10-15,False,False
3,2026-11-05,False,False
4,2026-09-20,False,False


### Output explanation
The flags can represent billing cycles, salary periods, or month-end purchasing patterns. They should be retained only if they are relevant to the business problem.

## 7. Days Since Event

Elapsed time measures how many days have passed between an event and a fixed reference date.

### Example
Calculate days since the last purchase as of the reference date.

In [7]:
orders["days_since_last_purchase"] = (
    reference_date - orders["last_purchase_date"]
).dt.days
orders[["last_purchase_date", "days_since_last_purchase"]]

,last_purchase_date,days_since_last_purchase
0,2026-11-01,4
1,2026-10-28,8
2,2026-10-15,21
3,2026-11-05,0
4,2026-09-20,46


### Output explanation
A smaller value means the customer purchased more recently. The reference date must match the model's prediction date, not an arbitrary future date.

## 8. Time Difference

Time difference measures the duration between two dates or timestamps.

### Example
Calculate days from signup to order.

In [8]:
orders["days_from_signup_to_order"] = (
    orders["order_date"] - orders["signup_date"]
).dt.days
orders[["signup_date", "order_date", "days_from_signup_to_order"]]

,signup_date,order_date,days_from_signup_to_order
0,2024-01-10,2026-11-01,1026
1,2023-06-20,2026-10-28,1226
2,2025-03-01,2026-10-15,593
3,2022-11-15,2026-11-05,1451
4,2026-01-05,2026-09-20,258


### Output explanation
This feature represents the elapsed time between account creation and an order. Check that both timestamps are valid and in the same timezone when time-of-day matters.

## 9. Customer Age

Customer age can be estimated from date of birth and a reference date. Dividing days by 365.25 is an approximation; a calendar-aware calculation is more accurate.

### Example
Calculate completed years of age.

In [9]:
def completed_years(start_dates, end_date):
    years = end_date.year - start_dates.dt.year
    before_birthday = (
        (end_date.month < start_dates.dt.month) |
        ((end_date.month == start_dates.dt.month) & (end_date.day < start_dates.dt.day))
    )
    return years - before_birthday.astype(int)

orders["customer_age"] = completed_years(
    orders["date_of_birth"], reference_date
)
orders[["date_of_birth", "customer_age"]]

,date_of_birth,customer_age
0,1998-04-12,28
1,1985-09-30,41
2,2001-01-20,25
3,1992-06-15,34
4,1978-12-02,47


### Output explanation
The calculation subtracts one year when the birthday has not yet occurred in the reference year. Use date-of-birth data only when it is appropriate, permitted, and relevant.

## 10. Customer Tenure

Tenure is the duration of a customer's relationship with a business as of a defined date.

### Example
Calculate completed days since signup.

In [10]:
orders["customer_tenure_days"] = (
    reference_date - orders["signup_date"]
).dt.days
orders[["signup_date", "customer_tenure_days"]]

,signup_date,customer_tenure_days
0,2024-01-10,1030
1,2023-06-20,1234
2,2025-03-01,614
3,2022-11-15,1451
4,2026-01-05,304


### Output explanation
A larger value means a longer customer relationship. The reference date should be the prediction cutoff so future information is not used.

## 11. Recency

Recency measures how recently a customer performed an action. In RFM analysis, purchase recency is often measured as days since the latest purchase.

### Example
Reuse the days-since-last-purchase feature as recency.

In [11]:
orders["recency_days"] = (
    reference_date - orders["last_purchase_date"]
).dt.days
orders[["customer_id", "recency_days"]]

,customer_id,recency_days
0,101,4
1,102,8
2,103,21
3,104,0
4,105,46


### Output explanation
Lower recency values indicate more recent activity. Recency is meaningful only when the last-purchase date is known as of the prediction cutoff.

## 12. Frequency

Frequency counts how many events occurred within a defined observation window. It normally requires transaction-level data, not just one row per customer.

### Example
Count orders per customer using a small transaction table.

In [12]:
transactions = pd.DataFrame({
    "customer_id": [101, 101, 102, 103, 101, 102, 104, 105, 103],
    "order_date": pd.to_datetime([
        "2026-08-01", "2026-09-10", "2026-08-15",
        "2026-09-01", "2026-10-20", "2026-10-02",
        "2026-10-12", "2026-09-22", "2026-10-29"
    ])
})

frequency = (
    transactions.groupby("customer_id")
    .size()
    .rename("order_frequency")
    .reset_index()
)
frequency

,customer_id,order_frequency
0,101,3
1,102,2
2,103,2
3,104,1
4,105,1


### Output explanation
The count gives the number of recorded transactions per customer in this sample. In a real model, restrict transactions to the observation window ending at the prediction cutoff.

## 13. Seasonality

Seasonality is a repeating pattern associated with a calendar period, such as month, quarter, weekday, or time of day.

### Example
Inspect the number of orders by month.

In [13]:
monthly_orders = (
    orders.groupby("order_month")
    .size()
    .reindex(range(1, 13), fill_value=0)
    .rename("order_count")
)
monthly_orders

order_month
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     1
10    2
11    2
12    0
Name: order_count, dtype: int64

### Output explanation
This summary shows how orders are distributed across months in the sample. A larger dataset covering multiple cycles is needed to establish a reliable seasonal pattern.

## 14. Cyclical Date Encoding

Calendar values such as month and weekday are cyclical: month 12 is close to month 1. Sine and cosine features can preserve this wrap-around relationship.

### Example
Encode month as sine and cosine values.

In [14]:
orders["month_sin"] = np.sin(2 * np.pi * orders["order_month"] / 12)
orders["month_cos"] = np.cos(2 * np.pi * orders["order_month"] / 12)
orders[["order_month", "month_sin", "month_cos"]]

,order_month,month_sin,month_cos
0,11,-0.500000,8.660254e-01
1,10,-0.866025,5.000000e-01
2,10,-0.866025,5.000000e-01
3,11,-0.500000,8.660254e-01
4,9,-1.000000,-1.836970e-16


### Output explanation
The two columns represent a month on a circle, so December and January are close in encoded space. This can help models learn repeating calendar patterns.